# NFL Play Prediction - Exploratory Data Analysis
**Author:** Joseph Sinder
**Institution:** University of Florida

## Overview
This notebook creates a basic Logistic Regression + LASSO model for NFL Play Prediction.

### Imports

In [2]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score

In [3]:
# Set Random Seed
RANDOM_SEED = 123

### Load Data

In [5]:
seasons = list(range(2016, 2026))
df = nfl.load_pbp(seasons).to_pandas()
part_df = nfl.load_participation(seasons).to_pandas()
df = df.merge(part_df[['play_id', 'offense_formation', 'offense_personnel', 'nflverse_game_id']], left_on=['play_id', 'game_id'], right_on=['play_id', 'nflverse_game_id'], how='left')

### Clean Data

In [8]:
# Only include plays that are either a run or pass
df_clean = df[df['play_type'].isin(['run', 'pass'])].copy()
# Extract RB count from personnel
df_clean['RB'] = df_clean['offense_personnel'].str.extract(r'(\d+) RB')[0].astype(float)
# Extract TE count from personnel
df_clean['TE'] = df_clean['offense_personnel'].str.extract(r'(\d+) TE')[0].astype(float)
# Extract WR count from personnel
df_clean['WR'] = df_clean['offense_personnel'].str.extract(r'(\d+) WR')[0].astype(float)
df_clean['play_type_val'] = (df_clean['play_type'] == 'pass').astype(int)
print(f"Data shape: {df_clean.shape}")

Data shape: (344820, 379)


### Features, Train/Test Split

In [9]:
NUM_FEATURES = ['RB', 'TE', 'WR', 'no_huddle', 'down', 'ydstogo', 'yardline_100', 'score_differential', 'goal_to_go', 'posteam_timeouts_remaining', 'defteam_timeouts_remaining', 'half_seconds_remaining', 'game_seconds_remaining', 'temp', 'wind']
CAT_FEATURES = ['offense_formation', 'roof']
FEATURES = NUM_FEATURES + CAT_FEATURES
# Create the model dataset
df_model = df_clean[FEATURES + ['play_type_val']].dropna()
# Split into train and test sets
X = df_model[FEATURES]
y = df_model['play_type_val']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

### Pipeline

In [22]:
# Create the preprocessing and modeling pipeline
num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipeline = OneHotEncoder(drop='first', handle_unknown='ignore')
# Create the full pipeline with preprocessing and logistic regression model
full_pipeline = ColumnTransformer([('num', num_pipeline, NUM_FEATURES), ('cat', cat_pipeline, CAT_FEATURES)])
# Create the final model pipeline
model = Pipeline([('preprocess', full_pipeline), ('log_reg', LogisticRegression(l1_ratio=1, solver='liblinear', random_state=RANDOM_SEED, max_iter=1000))])

### Fit and Evaluate

In [23]:
# Train the model and evaluate performance
model.fit(X_train, y_train)
# Evaluate the model on the test set
y_pred = model.predict(X_test)
# Calculate predicted probabilities
y_prob = model.predict_proba(X_test)[:, 1]
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")

Accuracy: 0.7266
ROC AUC: 0.7814
F1 Score: 0.7749
Precision: 0.7451
Recall: 0.8071
